# RASCALS / V4 detector upgrade

Correct the permanent alert latch, use an identical 60-second episode refresh for all baselines, and select a learned warning gate using validation only. Current breach limits bypass filtering. Final evaluation uses predeclared fresh 400000-series seeds after a model/protocol hash freeze. Prior v3 results remain development history.

In [1]:
from pathlib import Path
import os,sys,platform,json
ROOT=Path.cwd()
if not (ROOT/'continuityguard').is_dir():ROOT=ROOT.parent
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
assert platform.machine()=='s390x'
print(platform.machine(),platform.python_version())
OUT=ROOT/'artifacts/v4-candidate'

s390x 3.12.15


## Fit and select using training/calibration/validation only

In [2]:
from continuityguard.upgrade import fit_select
selected=fit_select(OUT)

Generating training, calibration and validation only; final test not generated.


Fit and validate logistic_C0.1


{"candidate": "logistic_C0.1", "best_validation": {"forecast_threshold": 0.98, "warning_gate": 0.5, "feasible_vs_static": true, "objective": 0.7825520833333334}, "precision": 0.4947916666666667, "early_recall": 0.9375}


Fit and validate logistic_C1


{"candidate": "logistic_C1", "best_validation": {"forecast_threshold": 0.98, "warning_gate": 0.8, "feasible_vs_static": true, "objective": 0.7823338963963964}, "precision": 0.5135135135135135, "early_recall": 0.9270833333333334}


...

Fit and validate boosted_trees_15_leaves


{"candidate": "boosted_trees_15_leaves", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.5, "feasible_vs_static": true, "objective": 0.784298155737705}, "precision": 0.5191256830601093, "early_recall": 0.9270833333333334}


Fit and validate boosted_trees_31_leaves


{"candidate": "boosted_trees_31_leaves", "best_validation": {"forecast_threshold": 1.01, "warning_gate": 0.5, "feasible_vs_static": true, "objective": 0.784298155737705}, "precision": 0.5191256830601093, "early_recall": 0.9270833333333334}


FROZEN before final evaluation: {"version": "cg-2026-10-17-v4", "trained_utc": "2026-10-17T13:01:41.109337+00:00", "architecture": "s390x", "model": "boosted_trees_15_leaves", "forecast_horizon_seconds": 60, "feature_count": 26, "sha256": "3a1525acbfeac41422d6b9d40ede39849050d53a61f7a94fbda194c54e4f3db2", "packages": {"numpy": "2.0.2", "scikit-learn": "1.9.0", "scipy": "1.13.1", "joblib": "1.6.0"}}


## Frozen fresh evaluation
The next cell verifies the previously written hashes before generating final runs. No policy adjustment follows final-test inspection.

In [3]:
from continuityguard.upgrade import evaluate_frozen
report=evaluate_frozen(OUT)
print('Frozen policy:',report['policy'])

Generating predeclared fresh final and stress runs AFTER freeze.


{
  "Static thresholds": {
    "incidents": 147,
    "alerts": 299,
    "detected": 143,
    "early_detections": 134,
    "late_detections": 9,
    "missed": 4,
    "incident_precision": 0.4782608695652174,
    "incident_recall": 0.9727891156462585,
    "early_warning_recall": 0.9115646258503401,
    "median_early_lead_seconds": 10.0,
    "healthy_service_hours": 30.0,
    "healthy_false_alerts": 0,
    "false_alarms_per_healthy_service_hour": 0.0,
    "unmatched_alerts_all_runs": 156
  },
  "Robust statistics": {
    "incidents": 147,
    "alerts": 1168,
    "detected": 145,
    "early_detections": 142,
    "late_detections": 3,
    "missed": 2,
    "incident_precision": 0.12414383561643835,
    "incident_recall": 0.9863945578231292,
    "early_warning_recall": 0.9659863945578231,
    "median_early_lead_seconds": 31.0,
    "healthy_service_hours": 30.0,
    "healthy_false_alerts": 29,
    "false_alarms_per_healthy_service_hour": 0.9666666666666667,
    "unmatched_alerts_all_runs": 102

Paired uncertainty: {"unit": "whole run; paired detector differences; simulator-conditional", "replicates": 500, "early_recall_delta_95": [-0.05795578488128202, -0.005796309987901574], "precision_delta_95": [0.02862660036887665, 0.0948972852340845]}


Frozen policy: {'mode': 'risk_filtered', 'forecast_threshold': 1.01, 'warning_gate': 0.5, 'novelty_threshold': 0.5967641153063866, 'novelty_quantile': 0.995, 'refresh_seconds': 60}
